# Blackboard

https://arxiv.org/pdf/2507.01701

## Board

In [1]:
from pydantic import BaseModel, Field, model_validator
from enum import Enum
import uuid

def get_id(n : int = 6):
    uuid4 = uuid.uuid4().hex[:n]
    return str(uuid4).replace('-', '')

class NoteStatus(Enum):
    ACTIVE='active'
    """Запись активна"""
    MARKED_FOR_DELETION='marked_for_deletion'
    """Помечена к удалению"""
    DELETED='deleted'
    """Заметка была удалена"""

class BaseNote(BaseModel):
    content : str = Field(max_length=2000, description='Содержимое записи')
    summary : str = Field(max_length=280, description='Суть записи в одном предложении')
    keywords : list[str] = Field(min_length=1, max_length=5, description='Ключевые слова')

class Note(BaseNote):
    id : str = Field(default='', description='ID записи')
    # status : NoteStatus = Field(default=NoteStatus.ACTIVE, description='Статус записи')
    author_id : str = Field(description='ID автора записи')

    @model_validator(mode='after')
    def _validate_model(self):
        self.id = get_id()
        return self

class Board(BaseModel):
    question : str = Field(description='Вопрос пользователя')
    notes : list[BaseNote] = Field(default=[], description='Список записей')

    def get_notes(self, last_n : int | None = None) -> list[dict]:
        """
        Возвращает список актуальных записей на доске с краткой информацией.
        Если передан last_n, вернет последние last_n записей.
        """
        notes = [{
            'id': note.id,
            'summary': note.summary,
            'keywords': note.keywords
        } for note in self.notes]
        if last_n is not None:
            notes = notes[-last_n:]
        return notes
    
    def get_note(self, note_id : str) -> Note | None:
        """
        Возвращает запись с указанным id.
        Возвращает None, если запись не найдена.
        """
        notes = [n for n in self.notes if n.id == note_id]
        if len(notes) == 0:
            return None
        return notes[0]
    
    def add_note(self, note : BaseNote, author_id : str) -> str:
        """
        Добавляет запись на доску.
        Возвращает id записи.
        """
        note = Note(author_id=author_id, **note.model_dump())
        self.notes.append(note)
        return note.id
    
    def remove_note(self, note_id : str):
        self.notes = [n for n in self.notes if n.id != note_id]

## Agents

In [2]:
from langchain.agents import create_agent
from langchain.tools import tool
from src import llm

In [3]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import BaseMessage

STRUCTURED_RESPONSE_KEY = 'structured_response'
MESSAGES_KEY = 'messages'

class Agent():

    def __init__(
        self,
        *args,
        id_ : str | None = None,
        tools : list | None = None,
        system_prompt : str | None = None,
        response_format : type[BaseModel] | None = None,
        checkpointer : InMemorySaver | None = None,
        summarization_tokens : int = 4000, 
        summarization_keep : int = 10,
        **kwargs
    ):
        self.id = id_ or get_id()
        
        self.tools = tools or []
        self.system_prompt = self._format_system_prompt(system_prompt)
        self.response_format = response_format
        self.checkpointer = checkpointer or InMemorySaver()

        summarization_middleware = SummarizationMiddleware(
            model=llm,
            trigger=("tokens", summarization_tokens),
            keep=("messages", summarization_keep)
        )
        
        self._agent = create_agent(
            *args,
            model=llm, 
            tools=tools, 
            response_format=response_format, 
            system_prompt=self.system_prompt,
            checkpointer=self.checkpointer,
            middleware=[summarization_middleware],
            **kwargs
        )

    def _format_system_prompt(self, system_prompt : str | None) -> str | None:
        return system_prompt

    def invoke(self, messages : BaseMessage, thread_id : str | None = None, force : bool = False):
        response = None

        def invoke():
            return self._agent.invoke(
                input={'messages': messages},
                config={"configurable": {"thread_id": thread_id or self.id}},
            )

        if force:
            while response is None:
                try:
                    response = invoke()
                except:
                    print('Maybe tool calls or something idk')
        else:
            response = invoke()
            
        if STRUCTURED_RESPONSE_KEY in response:
            return response[STRUCTURED_RESPONSE_KEY]
        return response[MESSAGES_KEY][-1].content
    
    @property
    def info(self):
        return {
            'id': self.id
        }
    
class Role(BaseModel):
    name : str = Field(description='Название роли')
    description : str = Field(max_length=140, description='Описание роли')
    
class RoleAgent(Agent):

    def __init__(self, role : Role, *args, **kwargs):
        self.role = role
        super().__init__(*args, **kwargs)

    def _format_system_prompt(self, system_prompt):
        if system_prompt is None:
            return system_prompt
        return system_prompt.format(**self.info)

    @property
    def info(self):
        return {
            **super().info,
            'role_name': self.role.name,
            'role_description': self.role.description
        }

### Generator

In [4]:
class GeneratorResponse(BaseModel):
    roles : list[Role] = Field(min_length=1, max_length=3, description='Список экспертных ролей')

_generator_prompt = """
Вам дан вопрос. Предоставьте мне список экспертных ролей, которые наиболее полезны для решения вопроса.
Вопрос:
{question}
"""

def create_generator_agent(board : Board) -> Agent:
    system_prompt = _generator_prompt.format(question=board.question)
    return Agent(system_prompt=system_prompt, response_format=GeneratorResponse)

### Expert

In [5]:
_id_prompt = """
Ваш личный id {id}.
"""
_system_prompt = """
Вы {role_name}, сотрудничающий с другими агентами для решения задачи.
Задача:
{question}
Существует общая доска, которую каждый из вас может читать и на которой может оставлять записи.
"""
_expert_prompt = """
Вы выдающийся специалист:
{role_name}
Описание:
{role_description}

Основываясь на ваших экспертных знаниях и текущем содержимом общей доски, решите задачу, изложите свои идеи и информацию, которую вы хотите записать на доску.
Совершенно не обязательно полностью соглашаться с точкой зрения, представленной на доске.
"""

def create_expert_agent(role : Role, board : Board, ):
    system_prompt = _id_prompt + (_system_prompt + _expert_prompt).format(
        question=board.question,
        role_name=role.name,
        role_description=role.description
    )
    agent = RoleAgent(
        role=role,
        system_prompt=system_prompt,
        tools=[
            tool(board.get_notes),
            tool(board.get_note)  
        ],
        response_format=BaseNote,
    )
    return agent

### Decider

In [6]:
class DeciderResponse(BaseModel):
    note : BaseNote = Field(description='Запись для добавления на доску')
    final_answer : str | None = Field(default=None, description='Окончательный развернутый ответ на задачу')

_decider_prompt = """
Ваша задача проанализировать текущее состояние общей доски и решить, достаточно ли у команды информации для получения окончательного ответа.
Если информации на доске достаточно для решения задачи, вы должны указать, что работа завершена, и предоставить окончательный ответ.
Если для получения ответа необходима дополнительная информация от других агентов, вы должны указать, что процесс следует продолжить.
"""

def create_decider_agent(board : Board):
    role = Role(
        name='Арбитр',
        description='Оценивает полноту информации. Выдает финальный ответ, либо инициирует продолжение обсуждения'
    )
    system_prompt = _id_prompt + (_system_prompt + _decider_prompt).format(
        role_name=role.name,
        question=board.question,
    )
    agent = RoleAgent(
        role=role,
        system_prompt=system_prompt,
        tools=[
            tool(board.get_notes),
            tool(board.get_note)  
        ],
        response_format=DeciderResponse,
    )
    return agent

### Planner

In [7]:
_planner_prompt = """
Создайте план решения исходной задачи на основе текущего содержимого общей доски.
Опишите задачу своими словами, затем изложите пошаговый план её решения.
Если план уже существует на доске или задача достаточно проста для прямого решения, просто укажите, что нет необходимости декомпозировать задачи и вы ожидаете дополнительной информации.
Не решайте задачу. Предоставьте только план.
"""

def create_planner_agent(board : Board):
    role = Role(
        name='Планировщик',
        description='Разрабатывает пошаговый план решения задачи на основе содержимого доски'
    )
    system_prompt = _id_prompt + (_system_prompt + _planner_prompt).format(
        role_name=role.name,
        question=board.question,
    )
    agent = RoleAgent(
        role=role,
        system_prompt=system_prompt,
        tools=[
            tool(board.get_notes),
            tool(board.get_note)  
        ],
        response_format=BaseNote,
    )
    return agent

### Critic

In [8]:
_critic_prompt = """
Проанализируйте записи на общей доске и определите любые бесполезные или избыточные.

Если вы обнаружите такие записи, опишите их и объясните, почему они должны быть удалены.
Если бесполезных записей нет, просто укажите, что бесполезных записей нет и вы ожидаете дополнительной информации.
"""

def create_critic_agent(board : Board):
    role = Role(
        name='Критик',
        description='Выявляет ошибочные или вводящие в заблуждение записи на доске'
    )
    system_prompt = _id_prompt + (_system_prompt + _critic_prompt).format(
        role_name=role.name,
        question=board.question,
    )
    agent = RoleAgent(
        role=role,
        system_prompt=system_prompt,
        tools=[
            tool(board.get_notes),
            tool(board.get_note)  
        ],
        response_format=BaseNote,
    )
    return agent

### Cleaner

In [9]:
_cleaner_prompt = """
Проанализируйте записи на общей доске и определите любые бесполезные или избыточные.

Если вы обнаружите такие записи:
- Перечислите их
- Для каждого объясните, почему оно бесполезно или избыточно

Если бесполезных записей нет, просто укажите, что бесполезных записей нет и вы ожидаете дополнительной информации.
"""

class CleanerResponse(BaseModel):
    note : BaseNote = Field(description='Запись для добавления на доску')
    notes_ids : list[str] = Field(default=[], description='Список ID записей к удалению')

def create_cleaner_agent(board : Board):
    role = Role(
        name='Уборщик',
        description='Анализирует доску, выявляет и удаляет бесполезные или избыточные записи'
    )
    system_prompt = _id_prompt + (_system_prompt + _cleaner_prompt).format(
        role_name=role.name,
        question=board.question,
    )
    agent = RoleAgent(
        role=role,
        system_prompt=system_prompt,
        tools=[
            tool(board.get_notes),
            tool(board.get_note)  
        ],
        response_format=CleanerResponse,
    )
    return agent

### Controller

In [10]:
class ControllerResponse(BaseModel):
    agents_ids : list[str] = Field(min_length=1, description='Упорядоченный список ID агентов')

_controller_prompt = """
Ваша задача назначить других агентов для сотрудничества и решения данной задачи.
Имена агентов и их описания перечислены ниже:
{agents}
Данная задача:
{question}
Агенты обмениваются информацией через общую доску.
Основываясь на содержимом, которое уже есть на доске, вам необходимо выбрать подходящих агентов из списка, чтобы они оставили записи на доске.
"""

def create_controller_agent(role_agents : list[RoleAgent], board : Board):
    system_prompt = _controller_prompt.format(
        agents=[rl.info for rl in role_agents],
        question=board.question
    )
    return Agent(
        system_prompt=system_prompt,
        tools=[
            tool(board.get_notes),
            tool(board.get_note)  
        ],
        response_format=ControllerResponse,
    )

In [11]:
def _get_agent(agent_id : str, agents : list[Agent]) -> Agent | None:
    for agent in agents:
        if agent.id == agent_id:
            return agent
    return None

def get_agents(agents_ids : list[str], agents : list[Agent]):
    result = []
    for agent_id in agents_ids:
        agent = _get_agent(agent_id, agents)
        if agent is not None:
            result.append(agent)
    return result

## Experiment

In [12]:
question = """
Целеполагание г .Гатчина (Ленинградская область) с 2026 по 2035 годы
"""

board = Board(question=question)

generator_agent = create_generator_agent(board)
roles = generator_agent.invoke([]).roles

In [13]:
expert_agents = [create_expert_agent(r, board) for r in roles]

for expert in expert_agents:
    print(expert.role.name, expert.role.description)

Градостроительный планировщик Разрабатывает генеральные планы развития территории, определяет зоны жилой, промышленной, рекреационной и инфраструктурной застройки, учитыв
Экономист‑стратег Проводит макро‑ и микроэкономический анализ, формирует финансовую модель развития города, оценивает инвестиционные потоки и доходность
Демографический аналитик Прогнозирует рост и структуру населения, миграционные потоки, потребности в жилье, образовании, здравоохранении и социальной защите


In [14]:
decider_agent = create_decider_agent(board)
planner_agent = create_planner_agent(board)
critic_agent = create_critic_agent(board)
cleaner_agent = create_cleaner_agent(board)

role_agents = [planner_agent, *expert_agents, critic_agent, cleaner_agent, decider_agent]
controller_agent = create_controller_agent(role_agents, board)

In [15]:
from tqdm import tqdm

final_answer = None

for i in range(10):
    print(f'Итерация {i+1}')
    agents_ids = controller_agent.invoke(
        [f'Записей на доске: {len(board.notes)}'], force=True
    ).agents_ids
    agents = get_agents(agents_ids, role_agents)
    print(str.join('\n', ['- ' + a.role.name for a in agents]))

    for a in agents:

        response = a.invoke([f'Записей на доске: {len(board.notes)}'], force=True)
        if a == decider_agent:
            final_answer = response.final_answer
            if final_answer is not None:
                break
            note = response.note
        elif a == cleaner_agent:
            notes_ids = response.notes_ids
            for note_id in notes_ids:
                board.remove_note(note_id)
            note = response.note
        else:
            note = response
        note_id = board.add_note(note, a.id)
        print(f'{note_id}, {a.role.name}: {note.summary}')

    if final_answer is not None:
        break

Итерация 1
- Планировщик
- Демографический аналитик
- Экономист‑стратег
- Градостроительный планировщик
8d41b8, Планировщик: Plan outline for the task.
b84c3c, Демографический аналитик: Демографический блок: прогноз населения, возрастная структура, миграция, потребности в жилье, образовании и здравоохранении для целей 2026‑2035.
e3ba28, Экономист‑стратег: Экономический блок: макро‑ и микроэкономический анализ, прогноз ВВП, структура отраслей, инвестиционный климат, финансовая модель бюджета Гатчины 2026‑2035.


Deserializing unregistered type __main__.ControllerResponse from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'ControllerResponse')]


fe5061, Градостроительный планировщик: Градостроительный блок: цели, KPI, зоны, транспорт, жильё, экология для Гатчины 2026‑2035.
Итерация 2


Deserializing unregistered type __main__.BaseNote from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'BaseNote')]


- Планировщик
- Критик
- Уборщик
- Арбитр
272a0e, Планировщик: Plan for the task based on board notes.
9422f2, Критик: Запись 272a0e дублирует 8d41b8 и должна быть удалена как избыточная.
8fdcd6, Уборщик: Избыточная запись


...

5de188, Демографический аналитик: Список всех 12 записей на доске с ID и кратким содержанием; указано, что запись 8d41b8 недоступна; предложены дальнейшие действия.


Deserializing unregistered type __main__.CleanerResponse from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'CleanerResponse')]


2014d3, Критик: Identified redundant notes: 75814b, d11a91, c5f7a2, 91bfc8, 659b17. Kept essential ones.


Deserializing unregistered type __main__.DeciderResponse from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'DeciderResponse')]


e3ccc0, Уборщик: Список бесполезных / избыточных записей


In [16]:
print(final_answer)

Информации на доске пока **недостаточно** для формирования окончательного, полностью обоснованного стратегического плана Гатчины 2026‑2035. Необходимо собрать и добавить в доску более детальные количественные прогнозы, финансовую модель бюджета, SMART‑цели и KPI для экономического и градостроительного блоков, а также конкретный перечень мероприятий с дорожной картой и механизмами мониторинга. Процесс следует продолжить, запросив эти данные у соответствующих агентов/источников.


In [21]:
print(response.note.content)

На доске сейчас **13 записей** (b84c3c, e3ba28, fe5061, d8c738, 101b5a, 447226, f9a696, 4b0c16, f8554a, b994d7, 5de188, 2014d3, e3ccc0).  

**Что уже есть**
- Общие демографические прогнозы и SMART‑цели (b84c3c, 101b5a).
- Макроэкономический прогноз, структура отраслей и упоминание финансовой модели (e3ba28).
- Краткое описание градостроительных направлений и экологических целей (fe5061, d8c738).
- Интегрированный «синергетический» блок (f8554a) с первичными количественными оценками (рост населения ≈ 202 000 чел., спрос ≈ 12 000 новых квартир, рост ВВП + 3,5 % год., инвестиции ≈ 200 млн руб., цели по озеленению 12 % и сокращению выбросов 20 %).
- Запись‑напоминание о недостатке данных (f9a696).

**Что всё‑ещё отсутствует для завершения стратегического плана 2026‑2035**
1. **Подробные количественные прогнозы** для всех блоков:
   - Возрастные когорты, миграционные потоки, потребности в школах, поликлиниках, соц‑инфраструктуре.
   - Секторный прогноз ВВП, доходов бюджета, налоговой базы,

In [19]:
print(note.content)

Анализируя 14 записей на общей доске, я выявил несколько, которые не добавляют новой информации или дублируют уже существующие материалы. Ниже перечислены эти записи и причины их избыточности/бесполезности.


In [18]:
board.notes

[Note(content='### Демографический блок для «Стратегического плана развития Гатчины\u202f2026‑2035»\n\n**1. Исходные демографические данные (2024\u202fг.)**\n| Показатель | Значение |\n|------------|----------|\n| Население (всего) | 92\u202f000 чел. |\n| Средний возраст | 41,2\u202fгода |\n| Коэффициент естественного прироста (ЧП) | –0,4\u202f% (отток) |\n| Чистый миграционный прирост (ЧМ) | +0,6\u202f% (приток) |\n| Доля населения 0‑14\u202fлет | 15\u202f% |\n| Доля населения 15‑64\u202fлет | 66\u202f% |\n| Доля населения 65+\u202fлет | 19\u202f% |\n| Уровень занятости | 71\u202f% (рабочая сила) |\n| Средняя численность домохозяйства | 2,4 чел. |\n\n**2. Прогноз демографических тенденций (2026‑2035)**\n- **Общее население**: рост до 96\u202f000\u202fчел. к\u202f2035\u202fг., среднегодовой темп +0,5\u202f% (за счёт миграции). \n- **Возрастная структура**: к\u202f2035\u202fг. доля 65+\u202fлет вырастет до 23\u202f% → увеличение спроса на услуги длительного ухода и адаптированное жильё.